In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
SILVER_TABLE_PATH = f"{SILVER_PATH}/price_table"
SILVER_TABLE_NAME = "vehicle_sales.silver.price_table"

In [0]:
price_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/Price_table")

In [0]:
price_df.display()

In [0]:
print(f"bronze row count: {price_df.count()}")

In [0]:
silver_price = (
    price_df
    .withColumn("Genmodel_ID", trim(col("Genmodel_ID")))
    .withColumn("Maker", trim(initcap(col("Maker"))))
    .withColumn("Genmodel", trim(col("Genmodel")))
    .withColumn("Year", col("Year").cast(IntegerType()))
    .withColumn("Entry_price", col("Entry_price").cast(DoubleType()))
    .filter(col("Genmodel_ID").isNotNull())
    .filter(col("Year").isNotNull())
    .filter((col("Entry_price").isNull()) | (col("Entry_price") >= 0))
    .dropDuplicates(["Genmodel_ID", "Year"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

In [0]:
silver_price.display()

####Data Quality Checks

In [0]:
row_count = silver_price.count()

In [0]:
null_key_count = silver_price.filter(col("Genmodel_ID").isNull() | col("Year").isNull()).count()

In [0]:
duplicate_key_count = silver_price.groupBy("Genmodel_ID", "Year").count().filter("count > 1").count()

In [0]:
negative_price_count = silver_price.filter(col("Entry_price") < 0).count()

In [0]:
print(f"silver row count: {row_count}")
print(f"null key count: {null_key_count}")
print(f"duplicate key count: {duplicate_key_count}")
print(f"negative Entry_price count: {negative_price_count}")

In [0]:
assert null_key_count == 0, "Genmodel_ID/Year should never be null in silver_price"
assert duplicate_key_count == 0, "(Genmodel_ID, Year) should be unique in silver_price"
assert negative_price_count == 0, "Entry_price should never be negative"

In [0]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
 
    silver_table = DeltaTable.forPath(spark, SILVER_TABLE_PATH)
 
    (silver_table.alias("t")
        .merge(silver_price.alias("s"), "t.Genmodel_ID = s.Genmodel_ID AND t.Year = s.Year")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    silver_price.write \
        .format("delta") \
        .mode("overwrite") \
        .save(SILVER_TABLE_PATH)
 


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME}
    USING DELTA
    LOCATION '{SILVER_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE_NAME} ZORDER BY (Genmodel_ID)")